# Cálculo de Rutas de Accesibilidad (1.25 km) para todas las Manzanas de la CDMX

Este notebook realiza el cálculo masivo de rutas sobre la **Red Nacional de Caminos (RNC)** para cada una de las manzanas de la Ciudad de México.

Genera una capa de rutas que represente el alcance de 1.25 km (1250 m) desde cada manzana, conservando la información socio-demográfica de la manzana y su ubicación por colonia y alcaldía.

In [1]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import MultiLineString, LineString
import numpy as np
from scipy.spatial import KDTree
import time
from pathlib import Path
import os
from tqdm.auto import tqdm

# Configuración de rutas de archivos
base_path = Path("..")
ruta_rnc = base_path / "datos/procesados/rnc_vial_cdmx.gpkg"
ruta_manzanas = base_path / "datos/procesados/manzanas_cdmx_poblacion.gpkg"
ruta_colonias = base_path / "datos/procesados/colonias_cdmx.gpkg"
ruta_salida = base_path / "datos/procesados/rutas_manzanas_cdmx_1250m.gpkg"

print("Configuración lista.")

Configuración lista.


### 1. Carga de Datos y Preprocesamiento

In [2]:
print("Cargando insumos...")
manzanas = gpd.read_file(ruta_manzanas).to_crs(epsg=32614)
colonias = gpd.read_file(ruta_colonias).to_crs(epsg=32614)
rnc = gpd.read_file(ruta_rnc).to_crs(epsg=32614)

print(f"Manzanas: {len(manzanas)}")
print(f"Colonias: {len(colonias)}")
print(f"Segmentos RNC: {len(rnc)}")

print("\nAsignando Colonia y Alcaldía a cada manzana...")
# Realizamos el join espacial. Usamos el centroide para mayor precisión en la asignación.
manzanas_centroides = manzanas.copy()
manzanas_centroides['geometry'] = manzanas.geometry.centroid

mza_info = gpd.sjoin(
    manzanas_centroides,
    colonias[['colonia', 'alc', 'geometry']],
    how="left",
    predicate="within"
)

# Eliminamos duplicados si alguna manzana cae en límites (usando CVEGEO)
mza_info = mza_info.drop_duplicates(subset=['CVEGEO'])

# Recuperamos la geometría original (polígono) de las manzanas
manzanas_full = manzanas.merge(
    mza_info[['CVEGEO', 'colonia', 'alc']], 
    on='CVEGEO', 
    how='left'
)

print("Preprocesamiento completado.")

Cargando insumos...
Manzanas: 63174
Colonias: 1543
Segmentos RNC: 113728

Asignando Colonia y Alcaldía a cada manzana...
Preprocesamiento completado.


### 2. Construcción del Grafo y KDTree

In [3]:
def build_graph(gdf_lines):
    G = nx.Graph()
    for idx, row in tqdm(gdf_lines.iterrows(), total=len(gdf_lines), desc="Construyendo grafo"):
        geom = row.geometry
        if isinstance(geom, LineString):
            coords = list(geom.coords)
            G.add_edge(coords[0], coords[-1], weight=geom.length, geometry=geom)
        elif isinstance(geom, MultiLineString):
            for line in geom.geoms:
                coords = list(line.coords)
                G.add_edge(coords[0], coords[-1], weight=line.length, geometry=line)
    return G

graph = build_graph(rnc)

print("Generando KDTree para búsqueda de nodos cercanos...")
nodes = list(graph.nodes)
# Nos aseguramos de usar solo X, Y para el KDTree
nodes_arr = np.array([(n[0], n[1]) for n in nodes])
tree = KDTree(nodes_arr)

print(f"Grafo listo con {len(graph.nodes)} nodos.")

Construyendo grafo:   0%|          | 0/113728 [00:00<?, ?it/s]

Generando KDTree para búsqueda de nodos cercanos...
Grafo listo con 81311 nodos.


### 3. Procesamiento Masivo de Distancias

In [4]:
def get_isodistance_geom(graph, tree, nodes, start_point, max_dist=1250):
    # Encontrar nodo más cercano
    _, idx = tree.query([start_point.x, start_point.y])
    start_node = nodes[idx]
    
    # Dijkstra con límite de distancia
    lengths = nx.single_source_dijkstra_path_length(graph, start_node, cutoff=max_dist, weight='weight')
    
    # Subgrafo y recolección de geometrías
    subgraph = graph.subgraph(lengths.keys())
    routes_geom = [data['geometry'] for u, v, data in subgraph.edges(data=True)]
    
    if not routes_geom:
        return None
    return MultiLineString(routes_geom)

resultados = []

# Usamos el centroide para el inicio de la ruta
for idx, mza in tqdm(manzanas_full.iterrows(), total=len(manzanas_full), desc="Procesando manzanas"):
    iso_geom = get_isodistance_geom(graph, tree, nodes, mza.geometry.centroid, max_dist=1250)
    
    if iso_geom:
        # Guardamos datos clave + nueva geometría de rutas
        resultados.append({
            'CVEGEO': mza['CVEGEO'],
            'colonia': mza['colonia'],
            'alcaldia': mza['alc'],
            'geometry': iso_geom
        })

gdf_rutas_final = gpd.GeoDataFrame(resultados, crs="EPSG:32614")
print(f"Procesamiento terminado. Se generaron rutas para {len(gdf_rutas_final)} manzanas.")

Procesando manzanas:   0%|          | 0/63174 [00:00<?, ?it/s]

Procesamiento terminado. Se generaron rutas para 63097 manzanas.


### 4. Guardado de Resultados

In [7]:
print(f"Guardando resultados en: {ruta_salida}")
gdf_rutas_final.to_file(ruta_salida, driver="GPKG")
print("Guardado exitoso.")

Guardando resultados en: ../datos/procesados/rutas_manzanas_cdmx_1250m.gpkg
Guardado exitoso.
